# Projeto 2 - Reinforcement Learning e ambientes não determinísticos

Este notebook implementa e compara Q-learning e SARSA no ambiente Taxi-v4 do Gymnasium, em cenários determinístico e não determinístico.

O projeto segue o enunciado da disciplina: cinco treinamentos por configuração, registro do reward por episódio, avaliação em 100 execuções e comparação por timesteps, desvio padrão, reward e taxa de sucesso.


## 1. Ambiente e hiperparâmetros

O Taxi-v4 possui 500 estados e 6 ações. O agente recebe -1 por passo, +20 ao entregar corretamente e -10 para pickup/dropoff ilegais.

No ambiente não determinístico usamos is_rainy=True e rainy_probability=0.6.

A meta de reward usada como referência nos gráficos é 8.


In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from rl_taxi import (
    EPISODES,
    ALPHA,
    GAMMA,
    EPSILON_START,
    EPSILON_MIN,
    EPSILON_DECAY,
    MAX_STEPS,
    SEEDS,
    train_many,
    evaluate_qtables,
    summarize_results,
    moving_average,
)

RESULTS_DIR = Path("resultados")
RESULTS_DIR.mkdir(exist_ok=True)

print("Episódios:", EPISODES)
print("Alpha:", ALPHA)
print("Gamma:", GAMMA)
print("Epsilon inicial:", EPSILON_START)
print("Epsilon mínimo:", EPSILON_MIN)
print("Epsilon decay:", EPSILON_DECAY)
print("Máximo de passos:", MAX_STEPS)
print("Seeds:", SEEDS)


## 2. Funções auxiliares para resultados e gráficos


In [ ]:
def save_rewards(filename, rewards):
    header = ",".join(f"run_{i + 1}" for i in range(rewards.shape[0]))
    np.savetxt(
        RESULTS_DIR / filename,
        rewards.T,
        delimiter=",",
        header=header,
        comments="",
        fmt="%.4f",
    )


def save_evaluations(filename, grouped_results):
    fields = ["condicao", "algoritmo", "q_table", "episode", "timesteps", "reward", "success"]

    with open(RESULTS_DIR / filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()

        for condition, algorithm, results in grouped_results:
            for row in results:
                writer.writerow(
                    {
                        "condicao": condition,
                        "algoritmo": algorithm,
                        **row,
                    }
                )


def smoothed_runs(rewards, window=100):
    return np.array([moving_average(run, window) for run in rewards])


def plot_learning(rewards_by_label, title, filename, window=100):
    plt.figure(figsize=(10, 5))

    for label, rewards in rewards_by_label.items():
        smooth = smoothed_runs(rewards, window)
        mean = smooth.mean(axis=0)
        std = smooth.std(axis=0)
        x = np.arange(window, window + len(mean))

        plt.plot(x, mean, label=label)
        plt.fill_between(x, mean - std, mean + std, alpha=0.15)

    plt.axhline(8, linestyle="--", label="Meta de reward (8)")
    plt.title(title)
    plt.xlabel("Episódio")
    plt.ylabel("Reward acumulado")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / filename, dpi=150)
    plt.show()


def first_target_episode(rewards, target=8, window=100):
    mean_curve = rewards.mean(axis=0)
    smooth = moving_average(mean_curve, window)
    indexes = np.where(smooth >= target)[0]
    if len(indexes) == 0:
        return None
    return int(indexes[0] + window)


def markdown_summary(summaries):
    lines = [
        "| Treinamento / avaliação | Timesteps médios | Desvio padrão | Reward médio | Taxa de sucesso |",
        "|---|---:|---:|---:|---:|",
    ]

    for label, summary in summaries.items():
        lines.append(
            f"| {label} | {summary['mean_timesteps']:.2f} | "
            f"{summary['std_timesteps']:.2f} | {summary['mean_reward']:.2f} | "
            f"{summary['success_rate']:.1f}% |"
        )

    display(Markdown("\n".join(lines)))


## 3. Q-learning e SARSA

Q-learning é off-policy e usa o maior valor Q disponível no próximo estado:

Q(s,a) ← Q(s,a) + α [r + γ max Q(s',a') - Q(s,a)]

SARSA é on-policy e atualiza usando a próxima ação escolhida pela própria política epsilon-greedy:

Q(s,a) ← Q(s,a) + α [r + γ Q(s',a') - Q(s,a)]

As implementações estão no arquivo rl_taxi.py.


## 4. Treinamento no ambiente determinístico

Cada algoritmo é treinado cinco vezes, usando as mesmas seeds e os mesmos hiperparâmetros.


In [ ]:
q_det_tables, q_det_rewards = train_many("q_learning", rainy=False)
s_det_tables, s_det_rewards = train_many("sarsa", rainy=False)

save_rewards("rewards_qlearning_deterministico.csv", q_det_rewards)
save_rewards("rewards_sarsa_deterministico.csv", s_det_rewards)

print("Treinamento determinístico concluído.")


## 5. Curvas de aprendizado no ambiente determinístico


In [ ]:
plot_learning(
    {"Q-learning": q_det_rewards},
    "Q-learning - ambiente determinístico",
    "qlearning_deterministico.png",
)

plot_learning(
    {"SARSA": s_det_rewards},
    "SARSA - ambiente determinístico",
    "sarsa_deterministico.png",
)

plot_learning(
    {
        "Q-learning": q_det_rewards,
        "SARSA": s_det_rewards,
    },
    "Q-learning x SARSA - ambiente determinístico",
    "comparacao_deterministico.png",
)


## 6. Avaliação no ambiente determinístico

Cada uma das cinco Q-tables é executada 20 vezes, totalizando 100 episódios de avaliação por algoritmo. Durante a avaliação não existe exploração e a Q-table não é atualizada.


In [ ]:
q_det_eval = evaluate_qtables(q_det_tables, rainy=False, episodes_per_table=20)
s_det_eval = evaluate_qtables(s_det_tables, rainy=False, episodes_per_table=20)

q_det_summary = summarize_results(q_det_eval)
s_det_summary = summarize_results(s_det_eval)

summaries = {
    "Q-learning: determinístico → determinístico": q_det_summary,
    "SARSA: determinístico → determinístico": s_det_summary,
}

markdown_summary(summaries)


## 7. Eficiência de aprendizado no ambiente determinístico


In [ ]:
q_target = first_target_episode(q_det_rewards)
s_target = first_target_episode(s_det_rewards)

print("Primeiro episódio em que a média móvel alcança reward 8:")
print("Q-learning:", q_target if q_target is not None else "não alcançou")
print("SARSA:", s_target if s_target is not None else "não alcançou")

if q_target is not None and s_target is not None:
    if q_target < s_target:
        print("Q-learning atingiu a meta antes nesta execução.")
    elif s_target < q_target:
        print("SARSA atingiu a meta antes nesta execução.")
    else:
        print("Os dois atingiram a meta no mesmo ponto aproximado.")
else:
    print("A comparação pela meta deve considerar que pelo menos um agente não a alcançou.")


## 8. Q-tables determinísticas executadas no ambiente não determinístico

Nesta etapa não há novo treinamento. As políticas aprendidas no ambiente determinístico são executadas com is_rainy=True e rainy_probability=0.6.


In [ ]:
q_det_rain_eval = evaluate_qtables(q_det_tables, rainy=True, episodes_per_table=20)
s_det_rain_eval = evaluate_qtables(s_det_tables, rainy=True, episodes_per_table=20)

q_det_rain_summary = summarize_results(q_det_rain_eval)
s_det_rain_summary = summarize_results(s_det_rain_eval)

summaries.update(
    {
        "Q-learning: determinístico → chuvoso": q_det_rain_summary,
        "SARSA: determinístico → chuvoso": s_det_rain_summary,
    }
)

markdown_summary(
    {
        "Q-learning: determinístico → determinístico": q_det_summary,
        "Q-learning: determinístico → chuvoso": q_det_rain_summary,
        "SARSA: determinístico → determinístico": s_det_summary,
        "SARSA: determinístico → chuvoso": s_det_rain_summary,
    }
)


### Discussão do impacto do não determinismo

A comparação acima mede diretamente se uma política treinada assumindo transições determinísticas continua eficiente quando os movimentos deixam de produzir sempre o resultado esperado. Devem ser observados principalmente aumento de timesteps, aumento de variabilidade, queda de reward e eventual queda na taxa de sucesso.


## 9. Treinamento no ambiente não determinístico

Agora Q-learning e SARSA são treinados novamente no próprio ambiente chuvoso, mantendo exatamente os mesmos hiperparâmetros.


In [ ]:
q_rain_tables, q_rain_rewards = train_many("q_learning", rainy=True)
s_rain_tables, s_rain_rewards = train_many("sarsa", rainy=True)

save_rewards("rewards_qlearning_chuvoso.csv", q_rain_rewards)
save_rewards("rewards_sarsa_chuvoso.csv", s_rain_rewards)

print("Treinamento não determinístico concluído.")


## 10. Curvas de aprendizado no ambiente não determinístico


In [ ]:
plot_learning(
    {
        "Q-learning": q_rain_rewards,
        "SARSA": s_rain_rewards,
    },
    "Q-learning x SARSA - ambiente não determinístico",
    "comparacao_chuvoso.png",
)


## 11. Avaliação após treinamento no ambiente não determinístico


In [ ]:
q_rain_eval = evaluate_qtables(q_rain_tables, rainy=True, episodes_per_table=20)
s_rain_eval = evaluate_qtables(s_rain_tables, rainy=True, episodes_per_table=20)

q_rain_summary = summarize_results(q_rain_eval)
s_rain_summary = summarize_results(s_rain_eval)

summaries.update(
    {
        "Q-learning: chuvoso → chuvoso": q_rain_summary,
        "SARSA: chuvoso → chuvoso": s_rain_summary,
    }
)

markdown_summary(summaries)


## 12. Salvando os resultados das 600 avaliações


In [ ]:
save_evaluations(
    "avaliacao.csv",
    [
        ("determinístico → determinístico", "Q-learning", q_det_eval),
        ("determinístico → determinístico", "SARSA", s_det_eval),
        ("determinístico → chuvoso", "Q-learning", q_det_rain_eval),
        ("determinístico → chuvoso", "SARSA", s_det_rain_eval),
        ("chuvoso → chuvoso", "Q-learning", q_rain_eval),
        ("chuvoso → chuvoso", "SARSA", s_rain_eval),
    ],
)

print("Resultados salvos em:", RESULTS_DIR.resolve())


## 13. Comparação final e conclusão


In [ ]:
def compare_value(name, a, b, smaller_is_better=True):
    if np.isclose(a, b):
        return f"{name}: resultados muito próximos ({a:.2f} e {b:.2f})."

    first_better = a < b if smaller_is_better else a > b
    winner = "Q-learning" if first_better else "SARSA"
    return f"{name}: {winner} apresentou o melhor valor nesta execução ({a:.2f} vs {b:.2f})."


display(Markdown("### Ambiente determinístico"))
print(compare_value(
    "Timesteps médios",
    q_det_summary["mean_timesteps"],
    s_det_summary["mean_timesteps"],
))
print(compare_value(
    "Reward médio",
    q_det_summary["mean_reward"],
    s_det_summary["mean_reward"],
    smaller_is_better=False,
))
print(
    f"Taxa de sucesso: Q-learning {q_det_summary['success_rate']:.1f}% | "
    f"SARSA {s_det_summary['success_rate']:.1f}%"
)

display(Markdown("### Política determinística testada no ambiente chuvoso"))
print(
    f"Q-learning: {q_det_summary['mean_timesteps']:.2f} → "
    f"{q_det_rain_summary['mean_timesteps']:.2f} timesteps médios."
)
print(
    f"SARSA: {s_det_summary['mean_timesteps']:.2f} → "
    f"{s_det_rain_summary['mean_timesteps']:.2f} timesteps médios."
)

display(Markdown("### Treinamento no ambiente chuvoso"))
print(compare_value(
    "Timesteps médios",
    q_rain_summary["mean_timesteps"],
    s_rain_summary["mean_timesteps"],
))
print(compare_value(
    "Reward médio",
    q_rain_summary["mean_reward"],
    s_rain_summary["mean_reward"],
    smaller_is_better=False,
))
print(
    f"Taxa de sucesso: Q-learning {q_rain_summary['success_rate']:.1f}% | "
    f"SARSA {s_rain_summary['success_rate']:.1f}%"
)


### Conclusão

As células anteriores geram os valores reais necessários para a discussão. A interpretação final deve considerar conjuntamente:

- velocidade de aprendizado nas curvas;
- reward acumulado;
- quantidade média de timesteps;
- desvio padrão;
- taxa de sucesso;
- efeito de executar uma política determinística em um ambiente não determinístico;
- diferença entre treinar diretamente no ambiente chuvoso e apenas transferir a política determinística.

Os resultados numéricos não são escritos manualmente neste notebook: eles são calculados a partir das execuções para evitar conclusões baseadas em valores inventados.
